In [1]:
import random
import torch
import os
import re

import pandas as pd
import polars as pl
import numpy as np

import sys
sys.path.append('../')
import compcor.corpus_metrics as corpus_metrics
from compcor.utils import Corpus
from compcor.text_tokenizer_embedder import STTokenizerEmbedder

from collections import defaultdict

/home/isabel/anaconda3/envs/localSyntheticData/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

In [2]:


# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)



In [3]:
# ------------------ Metric Setup (experiment_config.py) ------------------
comp_metrics = [
	corpus_metrics.chi_square_distance,
	corpus_metrics.zipf_distance,
	corpus_metrics.classifier_distance,
	corpus_metrics.IRPR_distance,
	corpus_metrics.fid_distance,
	corpus_metrics.pr_distance,
	corpus_metrics.dc_distance,
	corpus_metrics.mauve_distance,
	corpus_metrics.traditional_biber_distance,
	corpus_metrics.zero_wasserstein_distance
]

metrics_names = [((str(dist).split()[1]).split('_')[0]).upper() for dist in comp_metrics]

# Helper function for getting metric-dependent data.
def get_metric_dependant_data(metric, corpus: Corpus):
	if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
		c = STTokenizerEmbedder().tokenize_sentences(corpus)
	elif metric == corpus_metrics.zero_biber_distance:
		c = corpus
	else:
		c = STTokenizerEmbedder().embed_sentences(corpus)
	return c

# Helper function for getting all metric data.
def get_data_for_compcor_metrics(corpus):
	tokens = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").tokenize_sentences(corpus)
	embeddings = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").embed_sentences(corpus)
	return tokens, embeddings

def get_distances_from_compare_corpora(setA, setB):    
    tokensA, embeddingsA = get_data_for_compcor_metrics(setA)
    tokensB, embeddingsB = get_data_for_compcor_metrics(setB)
    distances = {}
    for metric_name, metric in zip(metrics_names, comp_metrics):
        if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
            tempA, tempB = tokensA, tokensB
        elif metric in (corpus_metrics.traditional_biber_distance, corpus_metrics.zero_wasserstein_distance):
            tempA, tempB = setA, setB
        else:
            tempA, tempB = embeddingsA, embeddingsB
        distances[metric_name] = metric(corpus1=tempA, corpus2=tempB)

    return distances


In [4]:
# subject the real data to the same processing as the generated data
def clean_note(text):
    text = re.sub(r'^[\s"]+|[\s"]+$', '', text)   # strip edge quotes/spaces
    text = re.sub(r'\*', '', text)                 # remove asterisks
    text = re.sub(r'\s+', ' ', text)              # normalize whitespace
    return text.strip()


temp_df = pd.read_csv(f'./combinedRealNotes/makeOneBigFile/dataRealAll.csv', index_col=None)
temp_df = temp_df.dropna(subset='Note')
temp_df['Note'] = [clean_note(text) for text in temp_df['Note'].tolist()]
real_dataset = temp_df

In [5]:
generated_datasets = {}
for dataset in os.listdir(f'./dataGeneration/processedData/'):
    if dataset.endswith('.csv'):
        temp_df = pd.read_csv(f'./dataGeneration/processedData/{dataset}', index_col=None)
        generated_datasets[dataset.replace('.csv', '')] = temp_df

In [6]:
dataset_metrics_averaged = {'allTopicModels': {}, 'LDA': {}, 'MATAVE': {}}
temp_models = {'allTopicModels': defaultdict(list), 'LDA': defaultdict(list), 'MATAVE': defaultdict(list)}
for i in range(3):
    real_sample = (real_dataset.sample(frac=1, random_state = i)['Note'].tolist())[:100]
    for model in temp_models:
        model_sample = (generated_datasets[model].sample(frac=1, random_state = i)['report'].dropna().tolist())[:100]
        temp_metrics = get_distances_from_compare_corpora(real_sample, model_sample)

        for metric, value in temp_metrics.items():
            temp_models[model][metric].append(value)

for model, metrics_dict in temp_models.items():
        for metric, values in metrics_dict.items():
            dataset_metrics_averaged[model][metric] = sum(values) / len(values)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

The `tokenize` method is deprecated, please use `preprocess` instead.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Num real: 100 Num fake: 100
Num real: 100 Num fake: 100


WARNING clustering 200 points to 10 centroids: please provide at least 390 training points
[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_07_second_person_pronouns', 'f_30_that_obj', 'f_32_wh_obj', 'f_35_because', 'f_47_hedges', 'f_50_discourse_particles', 'f_63_split_auxiliary']


In [7]:

dataset_metrics_averaged['realToReal'] = {}
temp_models = {'realToReal': defaultdict(list)}
for i in range(3):
    real_sample = real_dataset.sample(frac=1, random_state = i)['Note'].tolist()
    assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
    real_start = real_sample[:100]
    real_end = real_sample[-100:]
    temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

    for temp_metric, value in temp_metrics.items():
        temp_models['realToReal'][temp_metric].append(value)
                                                
for model, metrics_dict in temp_models.items():
        for temp_metric, values in metrics_dict.items():
            dataset_metrics_averaged[model][temp_metric] = sum(values) / len(values)


dataset_metrics_averaged['realToReal2'] = {}
temp_models = {'realToReal2': defaultdict(list)}
for i in range(3):
    real_sample = real_dataset.sample(frac=1, random_state = i + 200)['Note'].tolist()
    assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
    real_start = real_sample[:100]
    real_end = real_sample[-100:]
    temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

    for temp_metric, value in temp_metrics.items():
        temp_models['realToReal2'][temp_metric].append(value)
                                                
for model, metrics_dict in temp_models.items():
        for temp_metric, values in metrics_dict.items():
            dataset_metrics_averaged[model][temp_metric] = sum(values) / len(values)


dataset_metrics_averaged['realToReal3'] = {}
temp_models = {'realToReal3': defaultdict(list)}
for i in range(3):
    real_sample = real_dataset.sample(frac=1, random_state = i + 300)['Note'].tolist()
    assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
    real_start = real_sample[:100]
    real_end = real_sample[-100:]
    temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

    for temp_metric, value in temp_metrics.items():
        temp_models['realToReal3'][temp_metric].append(value)
                                                
for model, metrics_dict in temp_models.items():
        for temp_metric, values in metrics_dict.items():
            dataset_metrics_averaged[model][temp_metric] = sum(values) / len(values)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Num real: 100 Num fake: 100
Num real: 100 Num fake: 100


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Num real: 100 Num fake: 100
Num real: 100 Num fake: 100


WARNING clustering 200 points to 10 centroids: please provide at least 390 training points
[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_07_second_person_pronouns', 'f_12_proverb_do', 'f_13_wh_question', 'f_20_existential_there', 'f_22_that_adj_comp', 'f_23_wh_clause', 'f_29_that_subj', 'f_30_that_obj', 'f_32_wh_obj', 'f_33_pied_piping', 'f_34_sentence_relatives', 'f_35_because', 'f_36_though', 'f_47_hedges', 'f_59_contractions']


In [8]:
rows = []

for model, metrics in dataset_metrics_averaged.items():
    temp_dict = {
        'model': model,

    }
    for metric, value in metrics.items():
        temp_dict[metric] = value
    rows.append(temp_dict)

df = pd.DataFrame(rows)
df.to_csv("./ldaMataveMetrics.csv", index=False)

In [9]:
df

,model,CHI,ZIPF,CLASSIFIER,IRPR,FID,PR,DC,MAUVE,TRADITIONAL,ZERO
0,allTopicModels,0.000000,0.184044,0.967033,0.314002,0.759336,0.698251,0.932879,0.970704,0.510824,0.246050
1,LDA,0.000000,0.172124,0.951994,0.306521,0.726224,0.645994,0.915669,0.973749,0.504291,0.262112
2,MATAVE,0.000000,0.197774,0.983740,0.333860,0.837725,0.716517,0.941933,0.984797,0.518210,0.203755
3,realToReal,0.667949,0.027829,0.452498,0.193086,0.241809,0.117652,0.067015,0.030778,0.191097,0.091320
4,realToReal2,1.000000,0.047323,0.424715,0.189089,0.235171,0.112295,0.060020,0.015929,0.164203,0.113025
5,realToReal3,1.000000,0.012167,0.484539,0.190729,0.244031,0.127213,0.005042,0.021362,0.167957,0.098905
